# SmartAgri-AI: Exploratory Data Analysis (EDA)
**AI-Powered Crop Recommendation & Agricultural Decision Support System**

This notebook performs comprehensive exploratory analysis on the **Precision Agriculture Crop Recommendation Dataset** (2,200 samples across 22 crops).

### Features:
- **N, P, K**: Soil macronutrient ratios (kg/ha)
- **temperature**: Ambient temperature (°C)
- **humidity**: Relative atmospheric humidity (%)
- **ph**: Soil pH value
- **rainfall**: Precipitation (mm)
- **label**: Target crop species (22 distinct classes)

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_data, audit_dataset
from src.config import NUMERICAL_FEATURES, TARGET_COLUMN

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded successfully.")

## 1. Load Dataset and Audit Integrity

In [2]:
df = load_raw_data()
audit = audit_dataset(df)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing Values: {audit['total_missing_values']}")
print(f"Duplicate Rows: {audit['duplicate_rows']}")
print(f"Target Classes: {audit['num_target_classes']} crops (Balanced: {audit['is_balanced']})")
df.head()

## 2. Statistical Summary of Agricultural Attributes

In [3]:
df[NUMERICAL_FEATURES].describe().T

## 3. Target Class Balance Verification

In [4]:
plt.figure(figsize=(14, 5))
counts = df[TARGET_COLUMN].value_counts()
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False)
plt.xticks(rotation=45, ha="right")
plt.title("Crop Class Distribution (Exactly 100 observations per crop)", weight="bold")
plt.ylabel("Sample Count")
plt.show()

## 4. Pearson Correlation Analysis

In [5]:
plt.figure(figsize=(8, 6))
corr = df[NUMERICAL_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", square=True, vmin=-0.5, vmax=1.0)
plt.title("Correlation Heatmap of Agricultural Predictors", weight="bold")
plt.show()

## 5. Crop Agronomic Profiles (NPK Demands)

In [6]:
crop_profiles = df.groupby(TARGET_COLUMN)[NUMERICAL_FEATURES].median()
print("Top 5 Nitrogen-demanding crops:")
print(crop_profiles['N'].sort_values(ascending=False).head(5))

print("\nTop 5 Potassium-demanding crops:")
print(crop_profiles['K'].sort_values(ascending=False).head(5))

print("\nTop 5 Rainfall-dependent crops:")
print(crop_profiles['rainfall'].sort_values(ascending=False).head(5))